# Genotype–Phenotype Heterogeneity in Non-Classical 21-Hydroxylase Deficiency (FAIR^2) Exploration with `mlcroissant`
This notebook provides a guided walkthrough for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed (run once)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset package
dataset = mlc.Dataset(croissant_url)
# Access dataset metadata as a single object
metadata = dataset.metadata.to_json()

print(f"Dataset Title: {metadata['name']}")
print(f"Description: {metadata['description']}")

# Display dataset publication and identifier
print(f"Published: {metadata['datePublished']}")
print(f"Identifier: {metadata['identifier']}")
print(f"Keywords: {', '.join(metadata['keywords'])}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All entities are referenced by their Croissant `@id` fields.

In [ ]:
# Fetch Record Set IDs from the dataset metadata
# The RecordSet definitions are referenced by '@id' in the Croissant schema
record_set_ids = []
for record_set in dataset.record_sets():
    record_set_ids.append(record_set['@id'])
print("Available Record Sets (@id):")
for rsid in record_set_ids:
    print(f" - {rsid}")

# For each RecordSet, list its fields and their @id
for rsid in record_set_ids:
    print(f"\nRecord Set: {rsid}")
    fields = dataset.fields(record_set=rsid)
    for field in fields:
        print(f"  Field: {field['@id']} ({field['name']})")

# Show sample records from each RecordSet
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    print(f"\nSample record from {rsid}:")
    if records:
        print(records[0])

## 3. Data Extraction
Load data from each RecordSet into a DataFrame for analysis. Reference and utilize RecordSet and Field `@id`s.

In [ ]:
# Prepare a DataFrame for each RecordSet
dataframes = {}

for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"\nDataFrame Columns for {rsid}:")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. Filter records based on criteria, normalize numeric fields, categorize/group data.

We'll select a numeric field (e.g. hormone level, age) referenced by its `@id`, filter records, perform normalization, and group by categorical fields (e.g. phenotype, diagnosis status), all by their `@id`.

In [ ]:
# Example: Analyze Hormone levels from one RecordSet
# First, pick the relevant RecordSet @id and a numeric field @id
# (Replace with actual RecordSet and field @id from overview above)

# For this demo, we use the first RecordSet @id
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# List numeric fields (based on field data types)
fields = dataset.fields(record_set=record_set_id)
numeric_field_ids = []
for field in fields:
    if field.get('dataType') in ['Float', 'Integer', 'Number']:
        numeric_field_ids.append(field['@id'])

print(f"Numeric Fields (@id): {numeric_field_ids}")

if numeric_field_ids:
    numeric_field = numeric_field_ids[0]
    # Filter records: e.g., value > threshold
    threshold = 10
    if numeric_field in df.columns:
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"\nFiltered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Find groupable field (@id) (e.g., phenotype/diagnosis)
        group_field = None
        for field in fields:
            # Pick a categorical field
            if field.get('dataType') in ['Text', 'String']:
                group_field = field['@id']
                break
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
else:
    print("No numeric fields found for analysis in this RecordSet.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot histograms of selected numeric fields and (if possible) visualize relationships between numeric and categorical fields referenced by their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Visualize a numeric field's distribution
if numeric_field_ids:
    numeric_field = numeric_field_ids[0]
    if numeric_field in df.columns:
        plt.figure(figsize=(6,4))
        sns.histplot(df[numeric_field], kde=True)
        plt.title(f"Distribution of {numeric_field} in RecordSet {record_set_id}")
        plt.xlabel(numeric_field)
        plt.ylabel("Frequency")
        plt.show()

        # If group_field exists, plot boxplot
        if group_field and group_field in df.columns:
            plt.figure(figsize=(8,4))
            sns.boxplot(x=df[group_field], y=df[numeric_field])
            plt.title(f"{numeric_field} distribution by {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.show()


## 6. Conclusion
This notebook demonstrated how to load, overview, extract, process, and visualize the FAIR^2 data package using the `mlcroissant` library. All references to record sets, fields, and columns are made exclusively through their `@id`.

**Key insights:**
- The FAIR^2 dataset provides structured information on clinical and genetic features for non-classical 21-hydroxylase deficiency.
- Using `mlcroissant` with Croissant schemas ensures consistent referencing and interoperability for research workflows.
- This workflow is suitable for exploratory analysis, not for predictive ML model development, due to the dataset's small and specific cohort.

For further analysis, consider deeper phenotype-genotype correlations, or expanding the dataset with additional patient cohorts referenced by their `@id`.